In [ ]:
# NOTE: 原来的写法适配于 tensorflow 1.x，这里采用 tensorflow 2.x 的写法
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
import numpy as np

# 数据加载与处理
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train = x_train.reshape(-1, 28, 28, 1).astype(np.float32)
x_test = x_test.reshape(-1, 28, 28, 1).astype(np.float32)
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

# 这里保留的低级 API 函数，与原来题目相符
def conv2d(x, W):
    return tf.nn.conv2d(x, W, strides=[1,1,1,1], padding='SAME')

def max_pool_2x2(x):
    return tf.nn.max_pool(x, ksize=[1,2,2,1], strides=[1,2,2,1], padding='SAME')

# 初始化变量
def weight_variable(shape):
    return tf.Variable(tf.random.truncated_normal(shape, stddev=0.1))

def bias_variable(shape):
    return tf.Variable(tf.constant(0.1, shape=shape))

# 定义模型参数
W_conv1 = weight_variable([7, 7, 1, 32])
b_conv1 = bias_variable([32])

W_conv2 = weight_variable([5, 5, 32, 64])
b_conv2 = bias_variable([64])

W_fc1 = weight_variable([7*7*64, 1024])
b_fc1 = bias_variable([1024])

W_fc2 = weight_variable([1024, 10])
b_fc2 = bias_variable([10])

# 定义前向传播函数
def forward(x, training):
    x = conv2d(x, W_conv1) + b_conv1
    x = tf.nn.relu(x)
    x = max_pool_2x2(x)

    x = conv2d(x, W_conv2) + b_conv2
    x = tf.nn.relu(x)
    x = max_pool_2x2(x)

    x = tf.reshape(x, [-1, 7*7*64])
    x = tf.matmul(x, W_fc1) + b_fc1
    x = tf.nn.relu(x)
    if training:
        x = tf.nn.dropout(x, rate=0.3)
    x = tf.matmul(x, W_fc2) + b_fc2
    return tf.nn.softmax(x)

# 训练设置
learning_rate = 1e-4
optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
loss_fn = tf.keras.losses.CategoricalCrossentropy()

# 单步训练
@tf.function
def train_step(x, y):
    with tf.GradientTape() as tape:
        pred = forward(x, training=True)
        loss = loss_fn(y, pred)
    grads = tape.gradient(loss, [W_conv1, b_conv1, W_conv2, b_conv2, W_fc1, b_fc1, W_fc2, b_fc2])
    optimizer.apply_gradients(zip(grads, [W_conv1, b_conv1, W_conv2, b_conv2, W_fc1, b_fc1, W_fc2, b_fc2]))
    return loss

# 准确率评估函数
@tf.function
def compute_accuracy(x, y):
    pred = forward(x, training=False)
    correct = tf.equal(tf.argmax(pred, axis=1), tf.argmax(y, axis=1))
    return tf.reduce_mean(tf.cast(correct, tf.float32))

# 训练循环
batch_size = 100
max_steps = 2000

for step in range(max_steps):
    idx = np.random.choice(x_train.shape[0], batch_size)
    batch_x = x_train[idx]
    batch_y = y_train[idx]

    loss = train_step(batch_x, batch_y)

    if step % 100 == 0:
        acc = compute_accuracy(x_test[:1000], y_test[:1000])
        print(f"Step {step:4d}, Accuracy: {acc.numpy():.4f}")


Step    0, Accuracy: 0.2030
Step  100, Accuracy: 0.8840
Step  200, Accuracy: 0.9220
Step  300, Accuracy: 0.9350
Step  400, Accuracy: 0.9490
Step  500, Accuracy: 0.9550
Step  600, Accuracy: 0.9580
Step  700, Accuracy: 0.9580
Step  800, Accuracy: 0.9640
Step  900, Accuracy: 0.9660
Step 1000, Accuracy: 0.9670
Step 1100, Accuracy: 0.9720
Step 1200, Accuracy: 0.9640
Step 1300, Accuracy: 0.9660
Step 1400, Accuracy: 0.9690
Step 1500, Accuracy: 0.9730
Step 1600, Accuracy: 0.9710
Step 1700, Accuracy: 0.9700
Step 1800, Accuracy: 0.9690
Step 1900, Accuracy: 0.9700
